# Module 5: Reinforcement Learning - Tabular Q-Learning

Reinforcement Learning (RL) is a paradigm of machine learning where an **agent** learns to make decisions by performing actions in an **environment** to maximize some notion of cumulative **reward**.

In this notebook, we will explore:
1. **RL Terminology**: Agent, Environment, States ($S$), Actions ($A$), and Rewards ($R$).
2. **The Bellman Equation**: The mathematical foundation of Q-learning.
3. **Gymnasium Environment**: Setting up and interacting with a grid-world game called **FrozenLake**.
4. **Q-Table training**: Implementing the Q-learning algorithm from scratch (exploration vs. exploitation).

In [ ]:
import numpy as np
import gymnasium as gym
import random
import matplotlib.pyplot as plt

---

## 1. Q-Learning Intuition & The Bellman Equation

In tabular RL, the agent maintains a **Q-Table**, which has rows representing states and columns representing actions. The entry $Q(s, a)$ measures the "quality" of taking action $a$ in state $s$.

We update $Q(s, a)$ using the **Bellman Equation**:
$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ R + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

where:
- $\alpha$ is the **learning rate**.
- $\gamma$ is the **discount factor** (how much the agent values future rewards compared to immediate ones).
- $R$ is the reward received from state $s$ taking action $a$.
- $s'$ is the new state after taking action $a$.
- $\max_{a'} Q(s', a')$ is the estimated maximum future value from state $s'$.

---

## 2. Exploring the FrozenLake Environment

FrozenLake is a grid world where the agent wants to go from Start (S) to Goal (G) by walking on Frozen tiles (F) while avoiding Holes (H). 

- **Grid size**: 4x4 (16 states numbered 0 to 15).
- **Actions**: 4 directions (0: Left, 1: Down, 2: Right, 3: Up).
- **Reward**: +1 if the agent reaches the Goal, 0 otherwise.

In [ ]:
# Create the environment (is_slippery=False makes it deterministic for easier learning)
env = gym.make("FrozenLake-v1", is_slippery=False)

state_size = env.observation_space.n
action_size = env.action_space.n

print(f"Total States (S): {state_size}")
print(f"Total Actions (A): {action_size}")

# Reset to start state
state, info = env.reset()
print(f"Initial State: {state}")

---

## 3. Implementing the Q-Learning Algorithm

### Exploration vs. Exploitation: $\epsilon$-Greedy Policy
- **Exploration**: Taking random actions to discover new strategies ($"\epsilon"$ probability).
- **Exploitation**: Taking the best-known action in the Q-table to get the reward ($1 - \epsilon$ probability).

We decay $\epsilon$ over episodes as the agent gains experience.

In [ ]:
# Initialize Q-Table with zeros
q_table = np.zeros((state_size, action_size))

# Hyperparameters
total_episodes = 2000
learning_rate = 0.8
discount_factor = 0.95

# Exploration parameters
epsilon = 1.0
max_epsilon = 1.0
min_epsilon = 0.01
decay_rate = 0.005

rewards_history = []

# Training loop
for episode in range(total_episodes):
    state, info = env.reset()
    done = False
    episode_reward = 0
    
    while not done:
        # Choose action (epsilon-greedy)
        exp_tradeoff = random.uniform(0, 1)
        if exp_tradeoff > epsilon:
            # Exploit: best action
            action = np.argmax(q_table[state, :])
        else:
            # Explore: random action
            action = env.action_space.sample()
            
        # Take action, observe new state and reward
        new_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        
        # Update Q-Table using Bellman Equation
        q_table[state, action] = q_table[state, action] + learning_rate * (
            reward + discount_factor * np.max(q_table[new_state, :]) - q_table[state, action]
        )
        
        state = new_state
        episode_reward += reward
        
    # Decay epsilon
    epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episode)
    rewards_history.append(episode_reward)

print("Training finished!\n")
print("Final Q-Table:")
print(np.round(q_table, 3))

### Plotting Learning Progress
Let's plot rolling average rewards to see how the agent learned to solve the maze.

In [ ]:
def moving_average(a, n=100):
    ret = np.cumsum(a, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    return ret[n - 1:] / n

plt.figure(figsize=(10, 5))
plt.plot(moving_average(rewards_history, 100), color='green')
plt.title("Agent Success Rate (Moving Average over 100 Episodes)")
plt.xlabel("Episode")
plt.ylabel("Success Rate (Reaching Goal)")
plt.grid(True)
plt.show()

---

## 4. Test the Trained Agent

Now let's watch the agent navigate FrozenLake using only exploitation ($\epsilon = 0$).

In [ ]:
# Create renderable env
env_test = gym.make("FrozenLake-v1", is_slippery=False, render_mode="ansi")
state, info = env_test.reset()
done = False

print("Initial State:")
print(env_test.render())

step_count = 0
while not done:
    # Select best action (exploit)
    action = np.argmax(q_table[state, :])
    state, reward, terminated, truncated, info = env_test.step(action)
    done = terminated or truncated
    step_count += 1
    print(f"\nStep {step_count}: Action {action}")
    print(env_test.render())
    
env_test.close()

---

## 🏋️ Exercise: Slippery FrozenLake

By default, we set `is_slippery=False` which means moving right ALWAYS moves you right. 

In the real world, actions are stochastic (slippery): if you try to move right, there is a chance you slide left or down due to ice.

1. Re-create the environment using `gym.make("FrozenLake-v1", is_slippery=True)`.
2. Re-run training. Notice how the agent takes much longer to learn, and the success rate curves are noisier.
3. Adjust the `discount_factor` and `learning_rate` to see if you can improve performance.

In [ ]:
# YOUR CODE HERE